# AI-Powered Banking Customer Support & Ticket Intelligence

## Notebook 3 — Ticket Intelligence

This notebook extends the NovaBank AI customer-support system by
transforming customer conversations into structured support tickets.

The Ticket Intelligence module analyzes customer conversations and
automatically extracts:

- Customer intent
- Ticket category
- Sentiment
- Ticket priority
- AI-generated summary
- Suggested support action
- Structured ticket information

The goal is to reduce manual ticket-processing effort and provide
support teams with actionable information.

## 2. Reusing Previous Notebook Components

Instead of retraining models or manually copying information from
previous notebooks, the Ticket Intelligence module loads the artifacts
generated earlier in the project.

The following components are reused:

- `conversation_df` — customer conversation generated by Notebook 2
- `embedding_classifier` — semantic intent classifier trained in Notebook 1
- `embedding_model` — Sentence Transformer used for generating embeddings

This modular approach keeps the project organized and avoids
duplicating model-training code.

In [1]:
import pandas as pd

conversation_df = pd.read_csv(
    "../data/conversations.csv"
)

conversation_df

,conversation_id,user,assistant
0,C001,I lost my card,I'm so sorry to hear that you lost your NovaBa...
1,C001,I have frozen it,You've successfully frozen your card using the...
2,C001,Can I get a replacement?,You've already taken the necessary step to fre...


In [8]:
import joblib

embedding_classifier = joblib.load(
    "models/embedding_intent_classifier.pkl"
)

print("Model loaded successfully!")

from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

Model loaded successfully!


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
#Testing the classifier of Notebook(01_intent_classification_and_rag_retrieval)

test_query = conversation_df["user"].iloc[0]

query_embedding = embedding_model.encode(
    [test_query],
    normalize_embeddings=True
)

predicted_intent = embedding_classifier.predict(
    query_embedding
)[0]

print("Query:")
print(test_query)

print("\nPredicted Intent:")
print(predicted_intent)

Query:
I lost my card

Predicted Intent:
lost_or_stolen_card


## 3. Prepare Customer Conversation

A customer may send multiple messages during a support conversation.

For example:

Customer:
- "I lost my card"
- "I have frozen it"
- "Can I get a replacement?"

Instead of analyzing each message independently, we combine the
customer's messages into a single customer issue.

This combined text will be used as the input for downstream Ticket
Intelligence tasks.

In [12]:
customer_message = " ".join(
    conversation_df["user"].astype(str)
)

print("Customer Conversation:")
print(customer_message)

Customer Conversation:
I lost my card I have frozen it Can I get a replacement?


## 4. Customer Intent Classification

The intent classifier developed in Notebook 1 is reused to determine
the primary reason for the customer's support request.

The classifier uses Sentence Transformer embeddings rather than
traditional TF-IDF features.

The predicted intent provides a machine-readable representation of the
customer's issue and becomes one of the key fields in the support
ticket.

In [14]:
query_embedding = embedding_model.encode(
    [customer_message],
    normalize_embeddings=True
)

ticket_intent = embedding_classifier.predict(
    query_embedding
)[0]

print("Predicted Intent:", ticket_intent)

Predicted Intent: lost_or_stolen_card


## 5. Ticket Category Mapping

The intent classifier produces fine-grained intent labels such as
`lost_or_stolen_card` or `pending_transfer`.

For ticket management, these intents are grouped into broader
business categories such as:

- Cards
- Transfers
- Payments
- Verification
- Top Up
- Refunds
- Cash Withdrawal
- Currency

This makes tickets easier to organize, filter, and analyze.

In [15]:
intent_to_category = {
    "lost_or_stolen_card": "Cards",
    "card_not_working": "Cards",
    "card_activation": "Cards",
    "virtual_card": "Cards",
    "disposable_virtual_card": "Cards",

    "pending_transfer": "Transfers",
    "transfer_not_received_by_recipient": "Transfers",
    "bank_transfer": "Transfers",

    "card_payment_not_recognised": "Payments",
    "card_payment": "Payments",

    "unable_to_verify_identity": "Verification",
    "verify_my_identity": "Verification",

    "top_up_failed": "Top Up",
    "cash_withdrawal": "Cash Withdrawal",

    "exchange_rate": "Currency",
    "refund": "Refunds"
}

In [16]:
ticket_category = intent_to_category.get(
    ticket_intent,
    "General Support"
)

print("Ticket Category:", ticket_category)

Ticket Category: Cards


## 6. Sentiment Analysis

Sentiment analysis is used to identify the emotional tone of the
customer's message.

For the NovaBank support system, sentiment is classified into three
categories:

- Positive
- Neutral
- Negative

Customer sentiment is useful for ticket intelligence because highly
negative customer interactions may require faster attention from the
support team.

The sentiment score will later be combined with other signals such as
customer intent and issue severity to determine ticket priority.

### 6.1 Load Pre-trained Sentiment Model

A pre-trained Transformer model is used to analyze customer sentiment.

The model receives the customer's conversation as input and returns:

- Predicted sentiment label
- Confidence score

Using a pre-trained model allows us to perform sentiment analysis
without requiring a separate manually labelled NovaBank sentiment
dataset.

In [17]:
from transformers import pipeline

sentiment_analyzer = pipeline(
    "sentiment-analysis"
)

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

### 6.2 Analyze Customer Sentiment

The combined customer conversation created earlier is passed to the
sentiment analysis pipeline.

The model returns a sentiment label and its confidence score.

In [18]:
sentiment_result = sentiment_analyzer(
    customer_message
)

print(sentiment_result)

[{'label': 'NEGATIVE', 'score': 0.9997367262840271}]


### 6.3 Extract Sentiment Information

The sentiment model returns its prediction as a dictionary containing
the predicted label and confidence score.

We extract these values and store them as separate variables so they
can be used later when generating the support ticket and determining
ticket priority.

In [19]:
ticket_sentiment = sentiment_result[0]["label"]
sentiment_confidence = sentiment_result[0]["score"]

print("Sentiment:", ticket_sentiment)
print("Confidence:", round(sentiment_confidence, 4))

Sentiment: NEGATIVE
Confidence: 0.9997


### 6.4 Normalize Sentiment Label

The sentiment label is converted into a standardized format so that it
can be consistently stored in the final support ticket.

This makes the output easier to use in dashboards, filtering, and
downstream ticket-priority logic.

In [20]:
ticket_sentiment = ticket_sentiment.capitalize()

print("Ticket Sentiment:", ticket_sentiment)

Ticket Sentiment: Negative


## 6.5 Role of Sentiment in Ticket Intelligence

Sentiment is not used alone to determine ticket priority.

Instead, it will become one of several signals used by the
Ticket Intelligence system.

For example:

Customer Issue:
"Someone stole my card and I am extremely worried."

Intent:
lost_or_stolen_card

Sentiment:
Negative

Potential Priority:
High

In contrast:

Customer Issue:
"What exchange rate will I get for USD?"

Intent:
exchange_rate

Sentiment:
Neutral

Potential Priority:
Low

Therefore, sentiment provides additional context about the customer's
experience and urgency.

## 7.1 Priority Prediction

Ticket priority determines how urgently a customer issue should be
handled by the support team.

The priority engine combines:

1. Customer intent
2. Customer sentiment
3. Business-defined issue severity

A rule-based approach is used because support priority should be
transparent and explainable.

The system assigns one of three priority levels:

- HIGH
- MEDIUM
- LOW

### 7.2 High-Priority Issues

Certain customer issues require immediate attention because they may
involve security, unauthorized transactions, or loss of access.

Examples include:

- Lost or stolen cards
- Unrecognized card payments
- Cash withdrawals not recognized
- Identity verification problems

These issues are therefore considered high-priority candidates.

In [23]:
high_priority_intents = {
    "lost_or_stolen_card",
    "card_payment_not_recognised",
    "cash_withdrawal_not_recognised",
    "cash_withdrawal_not_recognised",
    "unable_to_verify_identity"
}

### 7.3 Medium-Priority Issues

Medium-priority issues are important but generally do not represent
an immediate security threat.

Examples include:

- Card not working
- Pending transfers
- Failed top-ups
- Transfer problems

In [24]:
medium_priority_intents = {
    "card_not_working",
    "pending_transfer",
    "transfer_not_received_by_recipient",
    "top_up_failed",
    "contactless_not_working",
    "top_up_reverted"
}

### 7.4 Priority Prediction Function

The priority function evaluates the predicted customer intent and
sentiment.

The decision logic is:

HIGH:
    Security-sensitive or potentially fraudulent issues.

MEDIUM:
    Operational issues that require support but are generally less
    urgent than security-related issues.

LOW:
    General informational requests or issues with limited urgency.

Negative sentiment can increase the urgency of operational issues,
but sentiment alone does not automatically make a ticket high priority.

In [25]:
def predict_priority(intent, sentiment):

    if intent in high_priority_intents:
        return "HIGH"

    elif intent in medium_priority_intents:
        if sentiment == "Negative":
            return "MEDIUM"
        else:
            return "MEDIUM"

    else:
        return "LOW"

In [26]:
ticket_priority = predict_priority(
    ticket_intent,
    ticket_sentiment
)

print("Ticket Priority:", ticket_priority)

Ticket Priority: HIGH


## 7.6 Current Ticket Intelligence

At this stage, the system has automatically extracted the core
intelligence from the customer conversation.

The information will later be combined with an AI-generated summary
and suggested support action to create the final structured ticket.

In [27]:
ticket_info = {
    "intent": ticket_intent,
    "category": ticket_category,
    "sentiment": ticket_sentiment,
    "sentiment_confidence": round(sentiment_confidence, 4),
    "priority": ticket_priority
}

ticket_info

{'intent': 'lost_or_stolen_card',
 'category': 'Cards',
 'sentiment': 'Negative',
 'sentiment_confidence': 0.9997,
 'priority': 'HIGH'}

## 8.1 AI-Powered Ticket Summarization

Customer conversations can contain multiple messages and unnecessary
conversation details.

The ticket summarization component uses the LLM to convert the
conversation into a concise description of the customer's primary
issue.

The summary should:

- Capture the main customer problem.
- Include important actions already taken by the customer.
- Preserve relevant context.
- Avoid adding information that was not provided.
- Be concise enough for a support agent to understand quickly.

The summary will become one of the main fields in the final support
ticket.

### 8.2 Ticket Summary Prompt

A dedicated prompt is used to instruct the LLM to generate a
support-ticket summary.

The prompt explicitly tells the model not to invent customer details
or NovaBank policies.

In [28]:
def create_ticket_summary_prompt(customer_message):

    prompt = f"""
You are NovaBank's AI ticket-intelligence assistant.

Summarize the following customer conversation for a support agent.

Requirements:
1. Clearly state the customer's main issue.
2. Include important actions already taken by the customer.
3. Keep the summary concise and professional.
4. Do not invent information.
5. Do not provide a solution or additional policy information.
6. Only summarize information explicitly present in the conversation.

Customer Conversation:
----------------------
{customer_message}
----------------------

Generate a concise support-ticket summary.
"""

    return prompt

### 8.3 Generate AI Ticket Summary

The customer's combined conversation is passed to the LLM using the
ticket-summary prompt.

The generated response becomes the `ticket_summary` field of the
support ticket.

In [1]:
import streamlit as st
from groq import Groq

client = Groq(api_key=st.secrets["GROQ_API_KEY"])

In [32]:
def generate_response(prompt):

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2,
        max_tokens=300
    )

    return response.choices[0].message.content

In [33]:
summary_prompt = create_ticket_summary_prompt(
    customer_message
)

ticket_summary = generate_response(
    summary_prompt
)

print("Ticket Summary:")
print(ticket_summary)

Ticket Summary:
The customer's main issue is that they have lost their card and are requesting a replacement. They have already taken the action of freezing their card.


### 8.4 Adding Summary to the ticket_info

In [35]:
ticket_info["summary"] = ticket_summary

ticket_info

{'intent': 'lost_or_stolen_card',
 'category': 'Cards',
 'sentiment': 'Negative',
 'sentiment_confidence': 0.9997,
 'priority': 'HIGH',
 'summary': "The customer's main issue is that they have lost their card and are requesting a replacement. They have already taken the action of freezing their card."}

## 8.5 Role of the LLM in Ticket Intelligence

The LLM is not responsible for determining the ticket intent or
priority.

Those components are handled by dedicated models and business rules.

The LLM is used for tasks where natural-language understanding and
generation are useful:

- Ticket summarization
- Suggested support action
- Customer-facing explanations

This separation makes the system more reliable and easier to evaluate.

Intent Classification → Machine Learning Model
Sentiment Analysis   → Pre-trained NLP Model
Priority Prediction  → Business Rules
Summary              → LLM
Suggested Action     → LLM

## 9.1 AI-Powered Suggested Support Action

The suggested-action component provides the support agent with a
recommended next step based on the customer's issue.

The recommendation is generated from the customer conversation and
the predicted ticket information.

The system should:

- Identify the most appropriate next action.
- Consider the customer's intent.
- Consider actions already taken by the customer.
- Avoid inventing NovaBank policies, fees, limits, or procedures.
- Provide a concise recommendation for the support agent.

This component is intended to assist support agents rather than
automatically make business decisions.

### 9.2 Suggested Action Prompt

The LLM receives the customer's conversation together with the
structured ticket information already extracted by the system.

Providing the intent, category, sentiment, and priority gives the LLM
additional context when generating the recommended support action.

In [36]:
def create_action_prompt(
    customer_message,
    intent,
    category,
    sentiment,
    priority
):

    prompt = f"""
You are NovaBank's AI ticket-intelligence assistant.

Recommend the next support action for the following customer ticket.

Customer Conversation:
----------------------
{customer_message}
----------------------

Ticket Information:
Intent: {intent}
Category: {category}
Sentiment: {sentiment}
Priority: {priority}

Requirements:
1. Recommend a practical next step for the support agent.
2. Consider what the customer has already done.
3. Keep the recommendation concise and professional.
4. Do not invent NovaBank policies, fees, limits, delivery times,
   or procedures.
5. Do not claim that an action has already been completed unless
   the customer explicitly said so.
6. If the available information is insufficient for a specific
   procedure, recommend that the support agent assist or verify
   the appropriate process.

Generate one concise suggested support action.
"""

    return prompt

### 9.3 Generate Suggested Support Action

The customer conversation and previously extracted ticket intelligence
are passed to the LLM.

The generated recommendation will become the `suggested_action`
field of the final support ticket.

In [37]:
action_prompt = create_action_prompt(
    customer_message,
    ticket_intent,
    ticket_category,
    ticket_sentiment,
    ticket_priority
)

suggested_action = generate_response(
    action_prompt
)

print("Suggested Action:")
print(suggested_action)

Suggested Action:
The support agent should assist the customer with the replacement card process, verifying their account information and any required details to facilitate the issuance of a new card.


### 9.4 Add It to ticket_info

In [38]:
ticket_info["suggested_action"] = suggested_action

ticket_info

{'intent': 'lost_or_stolen_card',
 'category': 'Cards',
 'sentiment': 'Negative',
 'sentiment_confidence': 0.9997,
 'priority': 'HIGH',
 'summary': "The customer's main issue is that they have lost their card and are requesting a replacement. They have already taken the action of freezing their card.",
 'suggested_action': 'The support agent should assist the customer with the replacement card process, verifying their account information and any required details to facilitate the issuance of a new card.'}

## 9.5 Summary vs Suggested Action

The Ticket Intelligence system separates the customer's situation
from the recommended response.

### Ticket Summary

Answers:

"What happened?"

Example:

Customer lost their card and has already frozen it. They are
requesting a replacement.

### Suggested Action

Answers:

"What should the support agent do next?"

Example:

Assist the customer with the appropriate card replacement process.

Keeping these fields separate makes the generated ticket more useful
for customer-support teams and allows each component to be evaluated
independently.

## 10.1 Creating the Final Support Ticket

The Ticket Intelligence module has now extracted multiple pieces of
information from the customer conversation.

These individual outputs are combined into a structured support ticket.

The final ticket contains:

- Ticket ID
- Customer conversation
- Intent
- Category
- Sentiment
- Sentiment confidence
- Priority
- Status
- AI-generated summary
- Suggested support action

A structured representation makes the ticket easier to store,
display in a dashboard, export to a database, or send to a support
management system.

### 10.2 Generate a Ticket ID

In [39]:
ticket_id = "NB-00001"

print("Ticket ID:", ticket_id)

Ticket ID: NB-00001


### 10.3 Ticket Status

Newly generated tickets are assigned the `Open` status.

The status can later be updated as the support team processes the
ticket.

In [40]:
ticket_status = "Open"

print("Ticket Status:", ticket_status)

Ticket Status: Open


### 10.4 Build Structured Support Ticket

All previously generated Ticket Intelligence outputs are combined into
a single Python dictionary.

This dictionary represents the final structured NovaBank support
ticket.

In [41]:
ticket = {
    "ticket_id": ticket_id,
    "customer_conversation": customer_message,
    "intent": ticket_intent,
    "category": ticket_category,
    "sentiment": ticket_sentiment,
    "sentiment_confidence": sentiment_confidence,
    "priority": ticket_priority,
    "status": ticket_status,
    "summary": ticket_summary,
    "suggested_action": suggested_action
}

ticket

{'ticket_id': 'NB-00001',
 'customer_conversation': 'I lost my card I have frozen it Can I get a replacement?',
 'intent': 'lost_or_stolen_card',
 'category': 'Cards',
 'sentiment': 'Negative',
 'sentiment_confidence': 0.9997367262840271,
 'priority': 'HIGH',
 'status': 'Open',
 'summary': "The customer's main issue is that they have lost their card and are requesting a replacement. They have already taken the action of freezing their card.",
 'suggested_action': 'The support agent should assist the customer with the replacement card process, verifying their account information and any required details to facilitate the issuance of a new card.'}

### 10.5 Display the Support Ticket

The structured ticket is displayed in a human-readable format so that
a support agent can quickly understand the customer's issue.

In [43]:
print("=" * 60)
print("           NOVABANK SUPPORT TICKET")
print("=" * 60)

print(f"Ticket ID       : {ticket['ticket_id']}")
print(f"Category        : {ticket['category']}")
print(f"Intent          : {ticket['intent']}")
print(f"Sentiment       : {ticket['sentiment']}")
print(f"Sentiment Score : {ticket['sentiment_confidence']:.4f}")
print(f"Priority        : {ticket['priority']}")
print(f"Status          : {ticket['status']}")

print("\nCustomer Conversation:")
print(ticket["customer_conversation"])

print("\nSummary:")
print(ticket["summary"])

print("\nSuggested Action:")
print(ticket["suggested_action"])

print("=" * 60)

           NOVABANK SUPPORT TICKET
Ticket ID       : NB-00001
Category        : Cards
Intent          : lost_or_stolen_card
Sentiment       : Negative
Sentiment Score : 0.9997
Priority        : HIGH
Status          : Open

Customer Conversation:
I lost my card I have frozen it Can I get a replacement?

Summary:
The customer's main issue is that they have lost their card and are requesting a replacement. They have already taken the action of freezing their card.

Suggested Action:
The support agent should assist the customer with the replacement card process, verifying their account information and any required details to facilitate the issuance of a new card.


## 10.6 Complete Ticket Intelligence Pipeline

The Ticket Intelligence module now processes a customer conversation
through multiple specialized components.

Customer Conversation
        ↓
Conversation Processing
        ↓
Intent Classification
        ↓
Category Mapping
        ↓
Sentiment Analysis
        ↓
Priority Prediction
        ↓
LLM Ticket Summarization
        ↓
LLM Suggested Action
        ↓
Structured Support Ticket

Different components are responsible for different tasks:

- Intent → Semantic ML classifier
- Category → Business mapping
- Sentiment → Pre-trained NLP model
- Priority → Explainable business rules
- Summary → LLM
- Suggested Action → LLM
- Ticket → Structured Python object

## 11.1 Convert Ticket into a DataFrame

The structured ticket generated by the Ticket Intelligence module is
currently stored as a Python dictionary.

For storing and analyzing multiple tickets, the ticket will be
converted into a Pandas DataFrame.

A tabular format makes it easier to:

- Store multiple tickets
- Filter tickets by priority
- Analyze customer sentiment
- Count tickets by category
- Build dashboards
- Export tickets to CSV

In [44]:
import pandas as pd

tickets_df = pd.DataFrame([ticket])

tickets_df

,ticket_id,customer_conversation,intent,category,sentiment,sentiment_confidence,priority,status,summary,suggested_action
0,NB-00001,I lost my card I have frozen it Can I get a re...,lost_or_stolen_card,Cards,Negative,0.999737,HIGH,Open,The customer's main issue is that they have lo...,The support agent should assist the customer w...


### 11.2 Inspect Ticket Dataset

Before saving the tickets, we inspect the structure of the dataset
to verify that all required ticket fields are present.

In [45]:
print("Number of tickets:", len(tickets_df))
print("Number of fields:", len(tickets_df.columns))

print("\nTicket fields:")
print(tickets_df.columns.tolist())

Number of tickets: 1
Number of fields: 10

Ticket fields:
['ticket_id', 'customer_conversation', 'intent', 'category', 'sentiment', 'sentiment_confidence', 'priority', 'status', 'summary', 'suggested_action']


## 11.3 Save Ticket Dataset

The generated ticket dataset is saved as a CSV file inside the
project's `data` directory.

This allows the Ticket Intelligence output to be reused by other
components of the project, such as:

- Streamlit dashboard
- Ticket analytics
- Evaluation
- Reporting
- Future database integration

In [46]:
tickets_df.to_csv(
    "data/tickets.csv",
    index=False
)

print("Tickets saved successfully.")

Tickets saved successfully.


### 11.4 Verify Saved Ticket Data

The saved CSV file is loaded again to verify that the ticket can be
successfully persisted and retrieved.

In [47]:
saved_tickets_df = pd.read_csv(
    "data/tickets.csv"
)

saved_tickets_df

,ticket_id,customer_conversation,intent,category,sentiment,sentiment_confidence,priority,status,summary,suggested_action
0,NB-00001,I lost my card I have frozen it Can I get a re...,lost_or_stolen_card,Cards,Negative,0.999737,HIGH,Open,The customer's main issue is that they have lo...,The support agent should assist the customer w...


In [48]:
print(
    "Number of saved tickets:",
    len(saved_tickets_df)
)

Number of saved tickets: 1


## 12.1 Sample Customer Conversations

To test the Ticket Intelligence pipeline on multiple cases, a small
collection of representative customer conversations is created.

These conversations simulate common NovaBank support scenarios such as:

- Lost or stolen cards
- Unrecognized card payments
- Pending transfers
- Failed top-ups
- Exchange-rate questions
- Card-not-working issues

These examples are used for pipeline testing and are not part of the
BANKING77 training dataset.

In [49]:
sample_conversations = [
    {
        "conversation_id": "C001",
        "customer_message": "I lost my card and I have already frozen it. Can I get a replacement?"
    },
    {
        "conversation_id": "C002",
        "customer_message": "I don't recognize a payment made with my card. I did not make this transaction."
    },
    {
        "conversation_id": "C003",
        "customer_message": "My bank transfer is still pending. It has been pending for a while."
    },
    {
        "conversation_id": "C004",
        "customer_message": "My top up failed and the money was not added to my account."
    },
    {
        "conversation_id": "C005",
        "customer_message": "What exchange rate will I get when I exchange USD to another currency?"
    },
    {
        "conversation_id": "C006",
        "customer_message": "My card is not working when I try to make a payment."
    }
]

sample_conversations_df = pd.DataFrame(
    sample_conversations
)

sample_conversations_df

,conversation_id,customer_message
0,C001,I lost my card and I have already frozen it. C...
1,C002,I don't recognize a payment made with my card....
2,C003,My bank transfer is still pending. It has been...
3,C004,My top up failed and the money was not added t...
4,C005,What exchange rate will I get when I exchange ...
5,C006,My card is not working when I try to make a pa...


### Why Use Multiple Test Conversations?

Testing multiple customer scenarios allows us to verify whether the
Ticket Intelligence pipeline behaves consistently across different
types of support requests.

For example:

Lost card
→ High priority

Unrecognized payment
→ High priority

Pending transfer
→ Medium priority

Failed top-up
→ Medium priority

Exchange-rate question
→ Low priority

Card not working
→ Medium priority

This provides a simple functional test of our ticket-processing logic.

## 12.3 Create a Reusable Ticket Processing Function

The Ticket Intelligence pipeline currently consists of several
individual processing steps.

To process multiple conversations efficiently, these steps are
combined into a reusable function.

The function receives a customer conversation and automatically:

1. Predicts the intent.
2. Determines the ticket category.
3. Performs sentiment analysis.
4. Predicts ticket priority.
5. Generates an AI summary.
6. Generates a suggested support action.
7. Creates a structured support ticket.

This makes the pipeline reusable for future conversations.

In [50]:
def process_ticket(
    customer_message,
    conversation_id,
    ticket_number
):

    # -----------------------------
    # 1. Intent Classification
    # -----------------------------

    query_embedding = embedding_model.encode(
        [customer_message],
        normalize_embeddings=True
    )

    intent = embedding_classifier.predict(
        query_embedding
    )[0]

    # -----------------------------
    # 2. Category
    # -----------------------------

    category = intent_to_category.get(
        intent,
        "General Support"
    )

    # -----------------------------
    # 3. Sentiment
    # -----------------------------

    sentiment_result = sentiment_analyzer(
        customer_message
    )

    sentiment = sentiment_result[0]["label"].capitalize()
    sentiment_confidence = sentiment_result[0]["score"]

    # -----------------------------
    # 4. Priority
    # -----------------------------

    priority = predict_priority(
        intent,
        sentiment
    )

    # -----------------------------
    # 5. AI Summary
    # -----------------------------

    summary_prompt = create_ticket_summary_prompt(
        customer_message
    )

    summary = generate_response(
        summary_prompt
    )

    # -----------------------------
    # 6. Suggested Action
    # -----------------------------

    action_prompt = create_action_prompt(
        customer_message,
        intent,
        category,
        sentiment,
        priority
    )

    suggested_action = generate_response(
        action_prompt
    )

    # -----------------------------
    # 7. Create Ticket
    # -----------------------------

    ticket = {
        "ticket_id": f"NB-{ticket_number:05d}",
        "conversation_id": conversation_id,
        "customer_conversation": customer_message,
        "intent": intent,
        "category": category,
        "sentiment": sentiment,
        "sentiment_confidence": round(
            sentiment_confidence,
            4
        ),
        "priority": priority,
        "status": "Open",
        "summary": summary,
        "suggested_action": suggested_action
    }

    return ticket

### 12.4 Test the Ticket Processing Function

Before processing the complete set of conversations, the reusable
pipeline is tested on a single customer conversation.

This helps verify that all Ticket Intelligence components are working
correctly inside the combined function.

In [51]:
test_ticket = process_ticket(
    customer_message=sample_conversations_df.loc[0, "customer_message"],
    conversation_id=sample_conversations_df.loc[0, "conversation_id"],
    ticket_number=2
)

test_ticket

{'ticket_id': 'NB-00002',
 'conversation_id': 'C001',
 'customer_conversation': 'I lost my card and I have already frozen it. Can I get a replacement?',
 'intent': 'lost_or_stolen_card',
 'category': 'Cards',
 'sentiment': 'Negative',
 'sentiment_confidence': 0.9997,
 'priority': 'HIGH',
 'status': 'Open',
 'summary': "The customer's main issue is that they lost their card and are requesting a replacement. They have already taken the action of freezing the lost card.",
 'suggested_action': 'The support agent should assist the customer with the replacement card process, verifying any required information to proceed with the request.'}

In [52]:
#Process Multiple Customer Tickets

processed_tickets = []

for index, row in sample_conversations_df.iterrows():

    ticket = process_ticket(
        customer_message=row["customer_message"],
        conversation_id=row["conversation_id"],
        ticket_number=index + 2
    )

    processed_tickets.append(ticket)


multi_ticket_df = pd.DataFrame(processed_tickets)

multi_ticket_df

,ticket_id,conversation_id,customer_conversation,intent,category,sentiment,sentiment_confidence,priority,status,summary,suggested_action
0,NB-00002,C001,I lost my card and I have already frozen it. C...,lost_or_stolen_card,Cards,Negative,0.9997,HIGH,Open,The customer's main issue is that they lost th...,The support agent should assist the customer w...
1,NB-00003,C002,I don't recognize a payment made with my card....,card_payment_not_recognised,Payments,Negative,0.9993,HIGH,Open,The customer is reporting an unrecognized paym...,The support agent should request more informat...
2,NB-00004,C003,My bank transfer is still pending. It has been...,pending_transfer,Transfers,Negative,0.9923,MEDIUM,Open,The customer's main issue is that their bank t...,The support agent should request additional de...
3,NB-00005,C004,My top up failed and the money was not added t...,top_up_reverted,General Support,Negative,0.9997,MEDIUM,Open,The customer's main issue is that their top-up...,Request the customer to provide the top-up tra...
4,NB-00006,C005,What exchange rate will I get when I exchange ...,exchange_rate,Currency,Negative,0.9987,LOW,Open,The customer is inquiring about the exchange r...,Request the customer to specify the target cur...
5,NB-00007,C006,My card is not working when I try to make a pa...,declined_card_payment,General Support,Negative,0.9994,LOW,Open,The customer's main issue is that their card i...,The support agent should ask the customer for ...


### 12.5 — Improve Intent → Category Mapping

In [55]:
intent_to_category.update({

    "top_up_reverted": "Top Up",
    "declined_card_payment": "Payments",

    "cash_withdrawal_not_recognised": "Cash Withdrawal",
    "cash_withdrawal": "Cash Withdrawal",

    "contactless_not_working": "Cards",
    "pending_card_payment": "Payments",

    "card_payment_wrong_exchange_rate": "Payments",
    "cash_withdrawal_wrong_exchange_rate": "Cash Withdrawal",

    "cash_withdrawal_not_recognised": "Cash Withdrawal",

    "balance_not_updated_after_bank_transfer": "Transfers",
    "balance_not_updated_after_cheque_or_cash_deposit": "Cash Withdrawal",

    "verify_my_identity": "Verification",
    "why_verify_identity": "Verification",
    "get_physical_card": "Cards",
    "get_disposable_virtual_card": "Cards"
})

In [56]:
processed_tickets = []

for index, row in sample_conversations_df.iterrows():

    ticket = process_ticket(
        customer_message=row["customer_message"],
        conversation_id=row["conversation_id"],
        ticket_number=index + 3
    )

    processed_tickets.append(ticket)


multi_ticket_df = pd.DataFrame(processed_tickets)

multi_ticket_df[
    [
        "ticket_id",
        "conversation_id",
        "intent",
        "category",
        "sentiment",
        "priority",
        "status"
    ]
]

,ticket_id,conversation_id,intent,category,sentiment,priority,status
0,NB-00003,C001,lost_or_stolen_card,Cards,Negative,HIGH,Open
1,NB-00004,C002,card_payment_not_recognised,Payments,Negative,HIGH,Open
2,NB-00005,C003,pending_transfer,Transfers,Negative,MEDIUM,Open
3,NB-00006,C004,top_up_reverted,Top Up,Negative,MEDIUM,Open
4,NB-00007,C005,exchange_rate,Currency,Negative,LOW,Open
5,NB-00008,C006,declined_card_payment,Payments,Negative,LOW,Open


## 12.6 Create an Analytics Symmary

In [59]:
analytics_summary = {
    "Total Tickets": len(multi_ticket_df),
    "High Priority Tickets": (
        multi_ticket_df["priority"] == "HIGH"
    ).sum(),
    "Medium Priority Tickets": (
        multi_ticket_df["priority"] == "MEDIUM"
    ).sum(),
    "Low Priority Tickets": (
        multi_ticket_df["priority"] == "LOW"
    ).sum(),
    "Open Tickets": (
        multi_ticket_df["status"] == "Open"
    ).sum(),
    "Most Common Category": (
        multi_ticket_df["category"].mode()[0]
    ),
    "Most Common Intent": (
        multi_ticket_df["intent"].mode()[0]
    )
}

analytics_summary

{'Total Tickets': 6,
 'High Priority Tickets': np.int64(2),
 'Medium Priority Tickets': np.int64(2),
 'Low Priority Tickets': np.int64(2),
 'Open Tickets': np.int64(6),
 'Most Common Category': 'Payments',
 'Most Common Intent': 'card_payment_not_recognised'}

### 12.7 Save Multi-Ticket Dataset

The processed tickets and their intelligence attributes are
saved as a CSV file.

This allows the Streamlit application to load previously
generated tickets instead of processing everything again.

In [61]:
multi_ticket_df.to_csv(
    "data/multi_tickets.csv",
    index=False
)

In [62]:
import os

os.path.exists("data/multi_tickets.csv")

True

In [65]:
saved_multi_ticket_df = pd.read_csv(
    "data/multi_tickets.csv"
)

saved_multi_ticket_df.shape

(6, 11)

## 12.7.1 Save Analytics Summary

In [63]:
analytics_df = pd.DataFrame(
    [analytics_summary]
)

analytics_df.to_csv(
    "data/ticket_analytics.csv",
    index=False
)

In [64]:
pd.read_csv(
    "data/ticket_analytics.csv"
)

,Total Tickets,High Priority Tickets,Medium Priority Tickets,Low Priority Tickets,Open Tickets,Most Common Category,Most Common Intent
0,6,2,2,2,6,Payments,card_payment_not_recognised


In [66]:
import os

os.makedirs(".streamlit", exist_ok=True)

print("Created:", os.path.abspath(".streamlit"))

Created: C:\Users\ashutosh yadav\AI-Powered Banking Customer Support\.streamlit


In [1]:
with open(".gitignore", "w") as f:
    f.write(".streamlit/secrets.toml\n")
    f.write(".env\n")

print(".gitignore created successfully.")

.gitignore created successfully.


In [2]:
with open(".gitignore", "r") as f:
    print(f.read())

.streamlit/secrets.toml
.env

